# Clean viruses → interim

Normalize multi-source virus detections onto modelling keys (spec `04`).

| Item | Detail |
|------|--------|
| Input | `data/raw/virus/virus_detections.tsv` + `countries.csv`, `subnational_locations.csv` |
| Output | `data/interim/viruses_normalized.parquet` |
| H3 | `H3_RESOLUTION` from `vector_guard.resolve` (locked = 4) |
| Time | `collection_date` → year-month (month required for panel keys) |
| Coords | Locked ladder: exact GPS / backfill → curated centroid → `country_only` |

**Guardrail:** never call external geocoders. Unmatched region strings stay `country_only` and are listed for optional `05` follow-up.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from vector_guard.data import (
    default_interim_dir,
    find_repo_root,
    load_virus_detections,
)
from vector_guard.resolve import (
    GEO_PRECISION_CENTROID,
    GEO_PRECISION_COUNTRY,
    GEO_PRECISION_EXACT,
    GEO_PRECISIONS,
    H3_RESOLUTION,
    assign_h3,
    build_name_to_iso2,
    build_subnational_centroid_index,
    load_countries,
    load_subnational_locations,
    lookup_subnational_centroid,
    norm_key,
    parse_year_month_virus,
    resolve_coords_ladder,
    resolve_virus_country_iso2,
)

REPO_ROOT = find_repo_root(Path.cwd())
INTERIM_DIR = default_interim_dir(REPO_ROOT)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = INTERIM_DIR / "viruses_normalized.parquet"

assert H3_RESOLUTION is not None and H3_RESOLUTION >= 0
REPO_ROOT, OUT_PATH, H3_RESOLUTION

(PosixPath('/workspaces/vector-guard'),
 PosixPath('/workspaces/vector-guard/data/interim/viruses_normalized.parquet'),
 4)

## Load raw + reference

In [2]:
raw = load_virus_detections()
countries = load_countries()
subnational = load_subnational_locations()

name_to_iso2 = build_name_to_iso2(countries)
centroid_index = build_subnational_centroid_index(subnational)

{
    "raw_rows": len(raw),
    "countries": len(countries),
    "subnational": len(subnational),
    "sources": sorted(raw["source"].fillna("").unique().tolist()),
}

{'raw_rows': 168277,
 'countries': 251,
 'subnational': 2472,
 'sources': ['bvbrc',
  'ecdc_atlas',
  'empres_i',
  'ena_read_run',
  'ena_sequence',
  'pathoplexus']}

## Resolve ISO-2, year-month, coords, H3

Coord priority (locked):

1. Valid `lat`/`lon` when `coord_source` ≠ `none` → `exact_coords`
2. `(country_iso2, subnational_region | nuts_id)` → `subnational_locations.csv` → `subnational_centroid`
3. Else → `country_only`, `h3_cell` null

In [3]:
def _text(value: object) -> str:
    text = str(value or "").strip()
    if text.lower() in {"nan", "none", "null", "na", "n/a", "<na>", "no_value"}:
        return ""
    return text


records = raw.to_dict(orient="records")

country_iso2 = [
    resolve_virus_country_iso2(row.get("country", ""), name_to_iso2) for row in records
]
subnational_region = [_text(row.get("subnational_region", "")) for row in records]
nuts_id = [_text(row.get("nuts_id", "")) for row in records]
coord_source = [_text(row.get("coord_source", "")).lower() for row in records]

year_month = [parse_year_month_virus(row.get("collection_date", "")) for row in records]
year = [pair[0] if pair else pd.NA for pair in year_month]
month = [pair[1] if pair else pd.NA for pair in year_month]

coords = [
    resolve_coords_ladder(
        lat=row.get("lat", ""),
        lon=row.get("lon", ""),
        coord_source=src,
        country_iso2=iso2,
        subnational_region=region,
        nuts_id=nuts,
        centroid_index=centroid_index,
        allow_centroid=True,
    )
    for row, src, iso2, region, nuts in zip(
        records, coord_source, country_iso2, subnational_region, nuts_id, strict=True
    )
]
lat = [item[0] for item in coords]
lon = [item[1] for item in coords]
geo_precision = [item[2] for item in coords]
h3_cell = [assign_h3(la, lo, H3_RESOLUTION) for la, lo in zip(lat, lon, strict=True)]

out = pd.DataFrame(
    {
        "record_id": [_text(row.get("record_id", "")) for row in records],
        "source": [_text(row.get("source", "")) for row in records],
        "pathogen_name": [_text(row.get("pathogen_name", "")) for row in records],
        "vector_group": [_text(row.get("vector_group", "")) for row in records],
        "country_iso2": country_iso2,
        "subnational_region": subnational_region,
        "nuts_id": nuts_id,
        "lat": lat,
        "lon": lon,
        "coord_source": coord_source,
        "geo_precision": geo_precision,
        "h3_cell": h3_cell,
        "year": year,
        "month": month,
        "accession": [_text(row.get("accession", "")) for row in records],
        "host": [_text(row.get("host", "")) for row in records],
    }
)

out["year"] = pd.to_numeric(out["year"], errors="coerce").astype("Int64")
out["month"] = pd.to_numeric(out["month"], errors="coerce").astype("Int64")

out.head()

,record_id,source,pathogen_name,vector_group,country_iso2,subnational_region,nuts_id,lat,lon,coord_source,geo_precision,h3_cell,year,month,accession,host
0,ena_sequence:AF206518,ena_sequence,West Nile virus,mosquito,US,"Connecticut, Greenwich-Stanford town line",,41.6486,-72.7594,,subnational_centroid,842a14bffffffff,<NA>,<NA>,AF206518,Culex pipiens
1,ena_sequence:AY187014,ena_sequence,West Nile virus,mosquito,US,"Harris County, TX",,NaN,NaN,,country_only,NaN,<NA>,<NA>,AY187014,
2,ena_sequence:AY278442,ena_sequence,West Nile virus,mosquito,RU,"Volgograd, low Volga",,49.5014,44.4488,,subnational_centroid,842d4d3ffffffff,<NA>,<NA>,AY278442,Human
3,ena_sequence:AY428518,ena_sequence,West Nile virus,mosquito,US,"Texas, Wichita county",,31.1310,-98.7607,,subnational_centroid,8448983ffffffff,<NA>,<NA>,AY428518,
4,ena_sequence:AY428519,ena_sequence,West Nile virus,mosquito,US,"Texas, Randall county",,31.1310,-98.7607,,subnational_centroid,8448983ffffffff,<NA>,<NA>,AY428519,


## QA coverage

In [4]:
n = len(out)
tier1 = out["h3_cell"].notna()
qa = {
    "rows": n,
    "iso2_mapped_share": float((out["country_iso2"] != "").mean()),
    "year_month_share": float((out["year"].notna() & out["month"].notna()).mean()),
    "vector_group_share": float((out["vector_group"] != "").mean()),
    "tier1_h3_share": float(tier1.mean()),
    "tier2_country_only_share": float((out["geo_precision"] == GEO_PRECISION_COUNTRY).mean()),
    "exact_among_tier1": float(
        (out.loc[tier1, "geo_precision"] == GEO_PRECISION_EXACT).mean()
    )
    if tier1.any()
    else float("nan"),
    "centroid_among_tier1": float(
        (out.loc[tier1, "geo_precision"] == GEO_PRECISION_CENTROID).mean()
    )
    if tier1.any()
    else float("nan"),
}
display(pd.Series(qa))

print("geo_precision")
display(out["geo_precision"].value_counts(dropna=False))
print("coord_source")
display(out["coord_source"].value_counts(dropna=False))
print("vector_group")
display(out["vector_group"].value_counts(dropna=False))

print("per-source geo")
display(
    out.groupby("source")
    .agg(
        rows=("record_id", "size"),
        iso2=("country_iso2", lambda s: (s != "").mean()),
        tier1=("h3_cell", lambda s: s.notna().mean()),
        ym=("year", lambda s: s.notna().mean()),
    )
    .sort_values("rows", ascending=False)
)

rows                        168277.000000
iso2_mapped_share                0.987550
year_month_share                 0.335928
vector_group_share               1.000000
tier1_h3_share                   0.380230
tier2_country_only_share         0.619770
exact_among_tier1                0.558733
centroid_among_tier1             0.441267
dtype: float64

geo_precision


geo_precision
country_only            104293
exact_coords             35750
subnational_centroid     28234
Name: count, dtype: int64

coord_source


coord_source
                   132527
exact_gps           34187
sample_backfill      1563
Name: count, dtype: int64

vector_group


vector_group
mosquito        110908
biting_midge     42432
tick             14230
sandfly            707
Name: count, dtype: int64

per-source geo


,rows,iso2,tier1,ym
source,,,,
ena_sequence,118381,0.993960,0.229353,0.116801
empres_i,25874,0.998686,1.000000,1.0
ena_read_run,13231,0.917013,0.482730,0.731388
pathoplexus,6377,0.991846,0.545711,0.647954
bvbrc,4108,0.952288,0.192795,0.677702
ecdc_atlas,306,1.000000,0.980392,0.767974


## Unmatched subnational / NUTS keys (no geocoding)

Rows with region text or `nuts_id` that did **not** resolve to a centroid and have no exact GPS. Feed candidates back into notebook `05` if coverage should improve.

In [5]:
unmatched_mask = (
    (out["geo_precision"] == GEO_PRECISION_COUNTRY)
    & (out["country_iso2"] != "")
    & ((out["subnational_region"] != "") | (out["nuts_id"] != ""))
)

unmatched_rows = out.loc[unmatched_mask].copy()


def region_lookup_key(row: pd.Series) -> str:
    if row["nuts_id"]:
        return norm_key(row["nuts_id"])
    region = row["subnational_region"]
    first = region.split(",", 1)[0].strip() if region else ""
    return norm_key(first)


unmatched_rows["region_key"] = unmatched_rows.apply(region_lookup_key, axis=1)
# Confirm still unmatched under the same lookup used by the ladder
still_unmatched = []
for row in unmatched_rows.itertuples(index=False):
    hit = lookup_subnational_centroid(
        row.country_iso2,
        subnational_region=row.subnational_region,
        nuts_id=row.nuts_id,
        centroid_index=centroid_index,
    )
    if hit is None and row.region_key:
        still_unmatched.append(
            {
                "country_iso2": row.country_iso2,
                "region_key": row.region_key,
                "subnational_region": row.subnational_region,
                "nuts_id": row.nuts_id,
                "source": row.source,
            }
        )

unmatched = (
    pd.DataFrame(still_unmatched)
    .value_counts(
        ["country_iso2", "region_key", "subnational_region", "nuts_id", "source"]
    )
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
    if still_unmatched
    else pd.DataFrame(
        columns=["country_iso2", "region_key", "subnational_region", "nuts_id", "source", "rows"]
    )
)

print(f"Unmatched region/NUTS keys with counts: {len(unmatched):,}")
display(unmatched.head(40))

Unmatched region/NUTS keys with counts: 4,295


,country_iso2,region_key,subnational_region,nuts_id,source,rows
0,VN,south,South,,ena_sequence,1216
1,CN,guangzhou,Guangzhou,,ena_sequence,674
2,VN,nha-trang,Nha Trang,,ena_sequence,562
3,CN,xi-shuang-ban-na,Xi Shuang Ban Na,,ena_sequence,259
4,VN,south,south,,ena_sequence,240
5,CN,mengla,"Mengla,Yunnan",,ena_sequence,175
6,CN,hangzhou,Hangzhou,,ena_sequence,171
7,FR,auvergne-allier,Auvergne - Allier,,ena_sequence,160
8,US,sacramento,"Sacramento, California",,ena_sequence,145
9,US,bakersfield,"Bakersfield,California",,ena_sequence,141


## Assert + write parquet

In [6]:
required_cols = [
    "record_id",
    "source",
    "pathogen_name",
    "vector_group",
    "country_iso2",
    "subnational_region",
    "nuts_id",
    "lat",
    "lon",
    "coord_source",
    "geo_precision",
    "h3_cell",
    "year",
    "month",
    "accession",
    "host",
]
assert list(out.columns) == required_cols
assert out["geo_precision"].isin(GEO_PRECISIONS).all()
assert out["record_id"].ne("").all()

has_coords = out["lat"].notna() & out["lon"].notna()
assert out.loc[has_coords, "lat"].between(-90, 90).all()
assert out.loc[has_coords, "lon"].between(-180, 180).all()
assert (
    out.loc[has_coords, "geo_precision"]
    .isin({GEO_PRECISION_EXACT, GEO_PRECISION_CENTROID})
    .all()
)
assert out.loc[has_coords, "h3_cell"].notna().all()
assert out.loc[~has_coords, "h3_cell"].isna().all()
assert out.loc[~has_coords, "geo_precision"].eq(GEO_PRECISION_COUNTRY).all()
assert (out.loc[out["month"].notna(), "month"].between(1, 12)).all()

# Exact GPS must win over centroid when both would apply
exact_gps_rows = out["coord_source"].isin({"exact_gps", "sample_backfill"}) & has_coords
assert out.loc[exact_gps_rows, "geo_precision"].eq(GEO_PRECISION_EXACT).all()

out.to_parquet(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH} ({len(out):,} rows)")
display(out["geo_precision"].value_counts())
display(out.head(10))
display(out.tail(5))

Wrote /workspaces/vector-guard/data/interim/viruses_normalized.parquet (168,277 rows)


geo_precision
country_only            104293
exact_coords             35750
subnational_centroid     28234
Name: count, dtype: int64

,record_id,source,pathogen_name,vector_group,country_iso2,subnational_region,nuts_id,lat,lon,coord_source,geo_precision,h3_cell,year,month,accession,host
0,ena_sequence:AF206518,ena_sequence,West Nile virus,mosquito,US,"Connecticut, Greenwich-Stanford town line",,41.6486,-72.7594,,subnational_centroid,842a14bffffffff,<NA>,<NA>,AF206518,Culex pipiens
1,ena_sequence:AY187014,ena_sequence,West Nile virus,mosquito,US,"Harris County, TX",,NaN,NaN,,country_only,NaN,<NA>,<NA>,AY187014,
2,ena_sequence:AY278442,ena_sequence,West Nile virus,mosquito,RU,"Volgograd, low Volga",,49.5014,44.4488,,subnational_centroid,842d4d3ffffffff,<NA>,<NA>,AY278442,Human
3,ena_sequence:AY428518,ena_sequence,West Nile virus,mosquito,US,"Texas, Wichita county",,31.1310,-98.7607,,subnational_centroid,8448983ffffffff,<NA>,<NA>,AY428518,
4,ena_sequence:AY428519,ena_sequence,West Nile virus,mosquito,US,"Texas, Randall county",,31.1310,-98.7607,,subnational_centroid,8448983ffffffff,<NA>,<NA>,AY428519,
5,ena_sequence:AY428522,ena_sequence,West Nile virus,mosquito,US,Illinois,,39.9460,-89.1991,,subnational_centroid,8426627ffffffff,<NA>,<NA>,AY428522,
6,ena_sequence:AY639639,ena_sequence,West Nile virus,mosquito,IN,,,NaN,NaN,,country_only,NaN,<NA>,<NA>,AY639639,Culex tritainiorhynchus
7,ena_sequence:AY701413,ena_sequence,West Nile virus,mosquito,MA,,,NaN,NaN,,country_only,NaN,<NA>,<NA>,AY701413,horse
8,ena_sequence:AY712948,ena_sequence,West Nile virus,mosquito,US,Texas,,31.1310,-98.7607,,subnational_centroid,8448983ffffffff,<NA>,<NA>,AY712948,Culex quinquefasciatus
9,ena_sequence:DQ010345,ena_sequence,West Nile virus,mosquito,US,"Suffolk Couunty, NY",,NaN,NaN,,country_only,NaN,<NA>,<NA>,DQ010345,American crow


,record_id,source,pathogen_name,vector_group,country_iso2,subnational_region,nuts_id,lat,lon,coord_source,geo_precision,h3_cell,year,month,accession,host
168272,ecdc_atlas:monthly:ES533:2026-W41,ecdc_atlas,West Nile virus,mosquito,ES,Menorca,ES533,39.950076,4.129265,,subnational_centroid,8439433ffffffff,<NA>,<NA>,,equid_outbreaks:1
168273,ecdc_atlas:monthly:ES611:2026-W41,ecdc_atlas,West Nile virus,mosquito,ES,Almería,ES611,37.210875,-2.312152,,subnational_centroid,8438243ffffffff,<NA>,<NA>,,equid_outbreaks:1
168274,ecdc_atlas:monthly:ES616:2026-W41,ecdc_atlas,West Nile virus,mosquito,ES,Jaén,ES616,38.017816,-3.444921,,subnational_centroid,84390a1ffffffff,<NA>,<NA>,,equid_outbreaks:1;bird_outbreaks:1
168275,ecdc_atlas:monthly:ES705:2026-W41,ecdc_atlas,West Nile virus,mosquito,ES,Gran Canaria,ES705,27.958743,-15.585429,,subnational_centroid,84344cbffffffff,<NA>,<NA>,,equid_outbreaks:3
168276,ecdc_atlas:monthly:CH033:2026-W41,ecdc_atlas,West Nile virus,mosquito,CH,Aargau,CH033,47.411178,8.154324,,subnational_centroid,841f83bffffffff,<NA>,<NA>,,equid_outbreaks:1
